# 2.2b Prompt engineering, model customization and retrieval tuning

**Story:** SafeDrive's claims assistant runs on an Amazon Bedrock foundation model (FM). You make it
do the job well in three steps: better prompts, (optionally) a customized model, and better retrieval.

**Task 2.2:** Train, fine-tune, and customize models for ML and AI solutions (GenAI part).

| Skill | Topic | Status |
|---|---|---|
| 2.2.8 | Customization: prompt engineering (Converse API, few-shot, JSON, tools, templates) | ✅ |
| 2.2.8 | Customization: fine-tuning job on Bedrock | 🔒 |
| 2.2.5 | Catastrophic forgetting (the FM part of this skill) | 📝 |
| 2.2.9 | Optimize retrieval components and embedding models | ✅ |

💰 A few hundred small Bedrock calls: well under USD 0.10. The fine-tuning cell is off by default.

## Key points for the exam

- **Converse API** = one request format for all Bedrock chat models: `system`, `messages`, `inferenceConfig`, `toolConfig`.
- **Temperature / topP** control randomness. Low (0 to 0.2) for extraction and classification; higher for creative text.
- **Zero-shot** = instruction only. **Few-shot** = instruction + examples. **Chain-of-thought** = ask the model to reason step by step.
- **Tool use**: the model does not run code. It returns a `toolUse` request; *your* code runs the tool and sends back a `toolResult`.
- Order of effort: prompt engineering → RAG → fine-tuning → continued pre-training. Try the cheap one first.
- **Fine-tuning** needs labeled pairs; **continued pre-training** needs unlabeled text; **distillation** uses a teacher model's answers.
- **Catastrophic forgetting**: after fine-tuning on a narrow dataset the model gets worse at what it knew before.
- Retrieval quality depends on **chunking**, the **embedding model and its dimension**, **top-k**, hybrid search, reranking and query rewriting. Measure with recall@k and MRR.

| Problem | First thing to try |
|---|---|
| Output format keeps changing | Few-shot examples, JSON schema in the prompt, or a forced tool call |
| Model lacks company facts | RAG |
| Right chunk is retrieved but ranked low | Reranker |
| Exact IDs or codes are missed | Hybrid (keyword + vector) search |

## Setup

In [ ]:
import sys; sys.path.append("..")
import json, re, time
import numpy as np, pandas as pd
from common import utils as U, catchup as C, rag
pd.set_option("display.max_colwidth", 140)
print(U.REGION, U.bucket())

Catch-up: raw data in S3 and the embedded policy chunks (both return at once if they exist).

In [ ]:
RAW = C.ensure_raw_data()
chunks = C.ensure_chunks()
brt = U.client("bedrock-runtime")
print(len(chunks), "policy chunks |", "text model:", U.TEXT_MODEL)

## 2.2.8 Customization, part 1: prompt engineering with the Converse API

- We call `bedrock-runtime.converse` directly here, because the API itself is the lesson.
- `system` sets the role and rules. `messages` is the conversation (`user` / `assistant` turns).
- The response carries the text, a `stopReason`, token `usage` and `metrics.latencyMs`.

### The basic call
A system prompt, one user message and an inference configuration. Look at the full shape of the response.

In [ ]:
resp = brt.converse(
    modelId=U.TEXT_MODEL,
    system=[{"text": "You are the SafeDrive Insurance claims assistant. Answer in one short sentence."}],
    messages=[{"role": "user", "content": [{"text": "What is a deductible?"}]}],
    inferenceConfig={"maxTokens": 100, "temperature": 0.2, "topP": 0.9})
print(resp["output"]["message"]["content"][0]["text"])
print("stopReason:", resp["stopReason"], "| usage:", resp["usage"], "| latency ms:", resp["metrics"]["latencyMs"])

A small helper for the rest of the notebook: it returns the text and the whole response.

In [ ]:
def chat(prompt, system=None, model_id=None, **config):
    args = dict(modelId=model_id or U.TEXT_MODEL,
                messages=[{"role": "user", "content": [{"text": prompt}]}],
                inferenceConfig={"maxTokens": 300, "temperature": 0, **config})
    if system:
        args["system"] = [{"text": system}]
    r = brt.converse(**args)
    return r["output"]["message"]["content"][0]["text"].strip(), r

chat("Say hello in five words.")[0]

**📘 Concept: Inference parameters (temperature, top-p, max tokens, stop sequences)**

A model produces text one token at a time. At each step it has a probability for every possible next token. The inference parameters change how one token is picked and when generation ends. They never change what the model knows.

| Parameter | What it does | Low value | High value |
|---|---|---|---|
| `temperature` | Flattens or sharpens the probabilities | Picks the most likely token almost every time: stable, repeatable | Less likely tokens get a real chance: varied, creative, more risk of errors |
| `topP` (nucleus sampling) | Keeps only the most likely tokens whose probabilities add up to P, then samples among them | Small candidate set: focused | Large candidate set: diverse |
| `maxTokens` | Hard limit on **output** tokens | Short answers, lower cost and latency; may cut the answer off | Room for long answers |
| `stopSequences` | Strings that end generation when produced | | |

- Usual advice: adjust temperature **or** top-p, not both at once.
- Output tokens drive both cost and latency, so `maxTokens` is also a cost control.
- Some models have further parameters (for example top-k); with the Converse API these go in `additionalModelRequestFields`.
- Exam cues: "consistent, factual, repeatable answers", "extraction", "classification" → temperature near 0. "Creative variety" → higher temperature or top-p. "Response is truncated" → raise `maxTokens`. "Model keeps writing after the answer" → stop sequence or lower `maxTokens`.

### `maxTokens` and `stopSequences`
`stopReason` tells you why the model stopped: `end_turn` (finished), `max_tokens` (cut off) or
`stop_sequence` (it produced one of your stop strings). Compare the three calls.

In [ ]:
question = "List five documents needed for a motor claim, one per line, numbered."
for label, cfg in [("default", {}), ("maxTokens=15", {"maxTokens": 15}), ("stop at '3.'", {"stopSequences": ["3."]})]:
    text, r = chat(question, **cfg)
    print(f"--- {label}: stopReason={r['stopReason']}, output tokens={r['usage']['outputTokens']}\n{text}\n")

### Temperature
Ask the same thing three times at temperature 0 and at temperature 1. Low temperature gives (almost)
the same answer each time; high temperature gives variety.

In [ ]:
ask = "Write a six-word slogan for a car insurance company."
for t in (0.0, 1.0):
    answers = [chat(ask, temperature=t, maxTokens=30)[0] for _ in range(3)]
    print(f"temperature={t}: {len(set(answers))} different answers\n  " + "\n  ".join(answers))

### Multi-turn conversation
The model has no memory. To continue a conversation you send the earlier turns again.

In [ ]:
history = [{"role": "user", "content": [{"text": "My claim is for a cracked windscreen."}]},
           {"role": "assistant", "content": [{"text": "Understood, a glass claim. How can I help?"}]},
           {"role": "user", "content": [{"text": "What type of claim did I just mention? One word."}]}]
r = brt.converse(modelId=U.TEXT_MODEL, messages=history, inferenceConfig={"maxTokens": 20, "temperature": 0})
print(r["output"]["message"]["content"][0]["text"])

### Zero-shot vs few-shot vs chain-of-thought
Task: decide if an adjuster note needs a fraud review. Load 20 notes. For this exercise the "true"
label is `REVIEW` when the note contains a red flag written by the adjuster (story changed, SIU, very new policy).

In [ ]:
notes = U.read_jsonl(f"{RAW}/adjuster_notes.jsonl")[:20]
for n in notes:
    n["label"] = "REVIEW" if ("SIU" in n["note"] or "three weeks ago" in n["note"]) else "ROUTINE"
pd.DataFrame(notes)[["note", "label"]].head(4)

Three prompt styles for the same task. Read them: the only difference is how much help the model gets.

In [ ]:
ZERO = "Classify the adjuster note as REVIEW (fraud red flags) or ROUTINE. Answer with one word.\n\nNote: {note}"

FEW = """Classify the adjuster note as REVIEW (fraud red flags) or ROUTINE. Answer with one word.

Note: Customer could not say where the car was parked. Policy started last week. High-value claim.
Label: REVIEW
Note: Small scratch on the door, photos match, repair shop estimate is normal.
Label: ROUTINE
Note: Vehicle reported stolen, no police report, customer gave two different dates.
Label: REVIEW

Note: {note}
Label:"""

COT = """Decide if the adjuster note needs a fraud review.
Think step by step: list any red flags (no police report, changing story, very new policy, estimate
near vehicle value). Then finish with a last line in the form 'LABEL: REVIEW' or 'LABEL: ROUTINE'.

Note: {note}"""
print(FEW.format(note=notes[0]["note"]))

Run one style over all notes and collect accuracy and token use. We read the label from the end of the answer.

In [ ]:
def run_style(template, max_tokens):
    correct = tokens_in = tokens_out = 0
    for n in notes:
        text, r = chat(template.format(note=n["note"]), model_id=U.SMALL_MODEL, maxTokens=max_tokens)
        found = re.findall(r"REVIEW|ROUTINE", text.upper())
        correct += bool(found) and found[-1] == n["label"]
        tokens_in += r["usage"]["inputTokens"]; tokens_out += r["usage"]["outputTokens"]
    return {"accuracy": correct / len(notes), "avg_in_tokens": tokens_in / len(notes),
            "avg_out_tokens": tokens_out / len(notes)}

run_style(ZERO, 5)

Compare the three styles on the small model. Look at accuracy **and** tokens: few-shot costs more input
tokens, chain-of-thought costs more output tokens (and latency).

In [ ]:
styles = {"zero-shot": (ZERO, 5), "few-shot": (FEW, 5), "chain-of-thought": (COT, 200)}
pd.DataFrame({name: run_style(t, m) for name, (t, m) in styles.items()}).T.round(2)

### Structured JSON output
Tell the model the exact keys you want and to return JSON only. Always parse and validate the result
in code: a model can still return broken JSON.

In [ ]:
EXTRACT = """Extract fields from the adjuster note. Return ONLY a JSON object with these keys:
"customer_name" (string or null), "policy_id" (string or null), "damage" (string, max 6 words),
"red_flags" (list of strings, empty if none), "needs_review" (true or false).

Note: {note}"""

def parse_json(text):
    """Take the first {...} block in the answer and parse it."""
    return json.loads(text[text.index("{"): text.rindex("}") + 1])

flagged = next(n for n in notes if n["label"] == "REVIEW")
raw, _ = chat(EXTRACT.format(note=flagged["note"]))
print(flagged["note"], "\n")
parse_json(raw)

### Tool use (function calling)
The model can ask for a function to be called. Here the tool is a **local** Python function that
looks up a claim. First load some claims and write the function.

In [ ]:
claims = U.read_csv(f"{RAW}/claims.csv").head(500).set_index("claim_id")

def get_claim_status(claim_id):
    if claim_id not in claims.index:
        return {"found": False, "claim_id": claim_id}
    row = claims.loc[claim_id]
    return {"found": True, "claim_id": claim_id, "claim_type": row["claim_type"],
            "claim_date": row["claim_date"], "claim_amount": float(row["claim_amount"])}

get_claim_status("C0000001")

Describe the tool to the model with a name, a description and a JSON schema for its input.
A clear description matters: the model uses it to decide *when* to call the tool.

In [ ]:
tool_config = {"tools": [{"toolSpec": {
    "name": "get_claim_status",
    "description": "Look up a SafeDrive claim by its claim ID and return type, date and amount.",
    "inputSchema": {"json": {
        "type": "object",
        "properties": {"claim_id": {"type": "string", "description": "Claim ID such as C0000001"}},
        "required": ["claim_id"]}}}}]}
print(json.dumps(tool_config, indent=1)[:300])

First call. The model cannot answer from its own knowledge, so it stops with `stopReason = tool_use`
and returns a `toolUse` block with the input it wants.

In [ ]:
messages = [{"role": "user", "content": [{"text": "How much was claimed on claim C0000002?"}]}]
r1 = brt.converse(modelId=U.TEXT_MODEL, messages=messages, toolConfig=tool_config,
                  inferenceConfig={"maxTokens": 300, "temperature": 0})
tool_use = next(c["toolUse"] for c in r1["output"]["message"]["content"] if "toolUse" in c)
print("stopReason:", r1["stopReason"])
tool_use

Your code runs the tool, then sends the result back as a `toolResult` in a `user` message
(after the assistant message that asked for it).

In [ ]:
result = get_claim_status(**tool_use["input"])
messages.append(r1["output"]["message"])
messages.append({"role": "user", "content": [{"toolResult": {
    "toolUseId": tool_use["toolUseId"], "content": [{"json": result}]}}]})
print(result)

Second call: the model now writes the final answer from the tool result.

In [ ]:
r2 = brt.converse(modelId=U.TEXT_MODEL, messages=messages, toolConfig=tool_config,
                  inferenceConfig={"maxTokens": 300, "temperature": 0})
print("stopReason:", r2["stopReason"])
print(r2["output"]["message"]["content"][0]["text"])

### Prompt templates
Keep the fixed part of a prompt in a template with named placeholders. You can then version it, test
it and reuse it. Bedrock **Prompt Management** stores such templates as versioned resources (used in 2.3b and 3.3b).

In [ ]:
TEMPLATE = """<role>You are the SafeDrive claims assistant.</role>
<rules>Answer only from the context. If the answer is not there, say "I don't know". Max {max_words} words.</rules>
<context>{context}</context>
<question>{question}</question>"""

prompt = TEMPLATE.format(max_words=25, context=chunks["text"].iloc[0], question="What does this section cover?")
print(chat(prompt)[0])

**Prompt engineering checklist**
- Give a role, clear rules and the output format. Put long context first and the question last.
- Use delimiters (XML-style tags) to separate instructions from data. This also reduces prompt injection risk.
- Add 2 to 5 good examples when the format or label set matters.
- Ask for step-by-step reasoning only when the task needs it: it costs tokens and time.
- Test prompts on an evaluation set (notebook 2.3b), not on one example.

## 2.2.8 Customization, part 2: fine-tuning, continued pre-training, distillation

| Method | Training data | Result | Bedrock `customizationType` |
|---|---|---|---|
| Fine-tuning | Labeled prompt → answer pairs (JSONL) | Model follows your style / task | `FINE_TUNING` |
| Continued pre-training | Unlabeled domain text | Model knows your domain language | `CONTINUED_PRE_TRAINING` |
| Distillation | Prompts (teacher model writes the answers) | Small, fast model close to teacher quality | `DISTILLATION` |

- **PEFT / LoRA**: freeze the base model and train small "adapter" matrices. Much cheaper, small artifacts,
  and the frozen base weights help against forgetting. On SageMaker AI you do this with JumpStart or the Hugging Face container.
- Fine-tuning hyperparameters: **epochs**, **learning rate**, warm-up steps, batch size. Too many epochs or too high a learning rate → overfitting and forgetting.
- A Bedrock custom model is served with **Provisioned Throughput** or (some models) on-demand custom model deployment.
- Keep a validation set and compare the tuned model with the base model on the same evaluation set.

**📘 Concept: Full fine-tuning vs PEFT / LoRA**

**Full fine-tuning** updates every weight of the model. It can change behaviour deeply, but it needs a lot of GPU memory and time, produces a complete new copy of the model for every task, and is the most likely to overwrite what the model already knew.

**Parameter-efficient fine-tuning (PEFT)** freezes the original weights and trains only a small number of new ones. **LoRA** (low-rank adaptation) is the common method: next to chosen weight matrices it adds two small matrices whose product is the "change" to apply. Only these are trained.

| | Full fine-tuning | LoRA / PEFT |
|---|---|---|
| Weights trained | All | A small fraction |
| Compute and time | High | Much lower |
| Artifact per task | A full model | A small adapter; one base model can serve many adapters |
| Risk of catastrophic forgetting | Higher | Lower, because the base weights are untouched |

- Related: **QLoRA** = LoRA on a quantized (smaller-precision) base model, to fit on smaller GPUs.
- Exam cues: "limited compute budget", "many tasks or customers on one base model", "avoid losing general capabilities" → PEFT / LoRA.

Load the cleaned fine-tuning file (`finetune_train_s3`, from notebooks 1.2b / 1.3). If it is not there
yet, apply a minimal clean-up to the raw file.

In [ ]:
ft_uri = U.get_state("finetune_train_s3")
if ft_uri:
    pairs = U.read_jsonl(ft_uri.split("/", 3)[3])
else:
    raw_rows = U.read_jsonl(f"{RAW}/finetune_qa.jsonl")
    pairs = [r for r in raw_rows if r.get("prompt") and r.get("completion") and len(r["completion"]) < 1000]
print(len(pairs), "training pairs, for example:", pairs[0])

Amazon Nova models expect the **conversation** format for fine-tuning (one JSON object per line with
`system` and `messages`). Convert the pairs and upload the file.

In [ ]:
SYSTEM = "You are the SafeDrive Insurance claims assistant. Answer briefly and exactly."
records = [{"schemaVersion": "bedrock-conversation-2024", "system": [{"text": SYSTEM}],
            "messages": [{"role": "user", "content": [{"text": p["prompt"]}]},
                         {"role": "assistant", "content": [{"text": p["completion"]}]}]}
           for p in pairs if p.get("prompt") and p.get("completion")]
FT_KEY = f"{U.PREFIX}/genai/finetune/train_nova.jsonl"
U.client("s3").put_object(Bucket=U.bucket(), Key=FT_KEY,
                          Body="\n".join(json.dumps(r) for r in records).encode())
print(len(records), "records at", U.s3_uri(FT_KEY))

🔒 **Fine-tuning job.** 💰 Costs money (training tokens, then monthly storage of the custom model; using
it needs Provisioned Throughput or a custom model deployment). Nova fine-tuning is available only in
some Regions (for example us-east-1). Set `U.RUN_EXPENSIVE = True` to run it.

In [ ]:
bedrock = U.client("bedrock")
FT_JOB = f"safedrive-ft-{int(time.time())}"
if U.expensive("Bedrock fine-tuning job"):
    job = bedrock.create_model_customization_job(
        jobName=FT_JOB, customModelName="safedrive-claims-nova-micro", roleArn=U.role_arn(),
        baseModelIdentifier="amazon.nova-micro-v1:0:128k", customizationType="FINE_TUNING",
        trainingDataConfig={"s3Uri": U.s3_uri(FT_KEY)},
        outputDataConfig={"s3Uri": U.s3_uri(f"{U.PREFIX}/genai/finetune/output/")},
        hyperParameters={"epochCount": "2", "learningRate": "0.00001", "learningRateWarmupSteps": "0"},
        jobTags=[{"key": t["Key"], "value": t["Value"]} for t in U.TAGS])
    print(job["jobArn"])

> 🖥️ **Console check:** If you ran this 🔒 cell: *Amazon Bedrock → Custom models → Jobs*.
> Look at the job status and its hyperparameters. When it completes, the model appears under the custom models list, where you choose how to serve it.

🔒 Check the job (it can take an hour or more). When it is `Completed`, the output folder in S3 holds
training metrics such as the loss per step.

In [ ]:
if U.expensive("fine-tuning job status"):
    j = bedrock.get_model_customization_job(jobIdentifier=FT_JOB)
    print(j["status"], j.get("failureMessage", ""), j.get("outputModelArn", ""))

For the other two methods only the request changes:

```python
# Continued pre-training: unlabeled text lines {"input": "..."} (supported by a few models)
customizationType="CONTINUED_PRE_TRAINING"

# Distillation: the student is baseModelIdentifier, the teacher is named in customizationConfig
customizationType="DISTILLATION",
customizationConfig={"distillationConfig": {"teacherModelConfig": {
    "teacherModelIdentifier": "<large model ARN>", "maxResponseLengthForInference": 1000}}}
```

## 2.2.5 Catastrophic forgetting 📝

**What it is:** after fine-tuning on a narrow dataset, the model becomes worse at tasks it handled well
before (general knowledge, other formats, safety behaviour). The new training overwrote useful weights.

| Prevention | How it helps |
|---|---|
| **PEFT / LoRA adapters** | Base weights stay frozen; only small adapters change |
| **Low learning rate, few epochs, early stopping** | Smaller changes to the weights |
| **Mix in general data** ("rehearsal" / replay) | Model keeps seeing the old tasks while learning the new one |
| **Freeze most layers** | Only the last layers adapt |
| **Regularization towards the old weights** (for example EWC) | Penalizes moving important weights |
| **Use RAG instead** | No weight change at all: nothing to forget |

- **Detect it:** evaluate the tuned model on a *general* benchmark as well as on your task, and compare with the base model.
- Overfitting vs forgetting: overfitting = bad on *new examples of the same task*; forgetting = bad on *other tasks it used to do*.
- Classic ML overfitting and underfitting are in notebook 2.2a.

## 2.2.9 Optimize retrieval components and embedding models

- Retrieval quality limits answer quality: the model cannot use a chunk it never receives.
- Things you can tune: **chunking** (fixed size, by heading, hierarchical, semantic), chunk size and overlap,
  the **embedding model** and its **dimension**, **top-k**, metadata filters, hybrid search, reranking, query rewriting.
- Measure with **recall@k** (is a relevant chunk in the top k?) and **MRR** (how high is the first relevant chunk? 1/rank, averaged).

Load the evaluation questions and the policy documents. Embed each question once (1024 dimensions).

In [ ]:
qa_set = U.read_jsonl(f"{RAW}/qa_eval.jsonl")
docs = {k.split("/")[-1]: U.read_bytes(k).decode() for k in U.list_keys(f"{RAW}/policy_docs/") if k.endswith(".md")}
q_vecs = np.array([U.embed(q["question"]) for q in qa_set])
print(len(qa_set), "questions,", len(docs), "documents, query matrix", q_vecs.shape)

We need to know which chunk is the *right* one for each question. Rule used here: the "gold" chunk is
the chunk from the question's source document that shares the most key words with the reference answer.

In [ ]:
def key_words(text):
    return {w for w in re.findall(r"[a-z0-9]+", text.lower()) if len(w) > 3 or w[0].isdigit()}

def gold_chunks(df, qa):
    """Positions of the best-matching chunk(s) for one question."""
    answer = key_words(qa["answer"])
    overlap = np.array([len(answer & key_words(t)) if s == qa["source"] else -1
                        for t, s in zip(df["text"], df["source"])])
    return set(np.flatnonzero(overlap == overlap.max()))

print(qa_set[0]["question"])
chunks["text"].iloc[list(gold_chunks(chunks, qa_set[0]))[0]]

The scoring function: for every question, rank all chunks by cosine similarity (`rag.top_k`), find the
position of the first gold chunk, and report recall@1, recall@3 and MRR.

In [ ]:
def evaluate(df, matrix, query_vectors):
    ranks = []
    for qa, qv in zip(qa_set, query_vectors):
        order, _ = rag.top_k(qv, matrix, k=len(df))
        gold = gold_chunks(df, qa)
        ranks.append(next(pos for pos, i in enumerate(order, start=1) if i in gold))
    ranks = np.array(ranks)
    return {"chunks": len(df), "recall@1": (ranks <= 1).mean(), "recall@3": (ranks <= 3).mean(),
            "mrr": (1 / ranks).mean()}

base_matrix = np.array(chunks["embedding"].tolist())
evaluate(chunks, base_matrix, q_vecs)

**📘 Concept: Chunking strategies**

Documents are split into chunks before embedding, because one vector can only represent a limited amount of meaning and the model's context should receive only the relevant parts. The strategy decides where the cuts fall.

| Strategy | How it splits | Good for | Watch out |
|---|---|---|---|
| **Fixed size** (with overlap) | Every N tokens; the overlap repeats the end of one chunk at the start of the next | Simple, uniform text; the usual starting point | Cuts through sentences and tables; overlap increases storage |
| **Hierarchical** (parent-child) | Small child chunks are searched, the larger parent is returned | Long structured documents where a short match needs its surroundings | More tokens sent to the model |
| **Semantic** | A model groups neighbouring sentences that are about the same thing | Text whose topics change without clear headings | Extra model cost at ingestion time |
| **None** | One chunk per file | Files you already split yourself (for example one FAQ per file) | Whole file must fit the embedding model's input limit |
| **Custom** | Your own logic, for example a Lambda function | Special formats (code, legal clauses) | You maintain it |

- These are the options you choose from in a Bedrock Knowledge Base data source. The strategy cannot be changed on an existing data source: you create a new one and re-ingest.
- Exam cues: "answers lack surrounding context" → larger chunks or hierarchical. "Retrieved chunks contain a lot of unrelated text" → smaller or semantic chunks. "Documents are already split into self-contained files" → no chunking.

### Chunking strategy and chunk size
Build three fixed-size chunkings (small, medium, large) and embed them. The heading-based chunks from
`C.ensure_chunks` are the fourth option.

In [ ]:
def build_fixed(size, overlap):
    rows = [c for name, text in docs.items()
            for c in rag.chunk_fixed(text, name, size=size, overlap=overlap) if c["text"].strip()]
    df = pd.DataFrame(rows)
    return df, np.array([U.embed(t) for t in df["text"]])

variants = {"by heading": (chunks, base_matrix)}
for size, overlap in [(150, 0), (400, 50), (1000, 100)]:
    variants[f"fixed {size} chars, overlap {overlap}"] = build_fixed(size, overlap)
{name: len(df) for name, (df, _) in variants.items()}

Compare. Very small chunks cut sentences and lose context; very large chunks mix many topics in one
vector (and send more tokens to the model). Chunks that follow the document structure usually do well.

In [ ]:
chunk_cmp = pd.DataFrame({name: evaluate(df, m, q_vecs) for name, (df, m) in variants.items()}).T
chunk_cmp.round(3)

### Embedding dimension: 256 vs 1024
Titan Text Embeddings V2 can return 256, 512 or 1024 dimensions. Smaller vectors need less storage
and search faster, with a small loss of accuracy. Re-embed the heading chunks and the questions at 256.

In [ ]:
m256 = np.array([U.embed(t, dimensions=256) for t in chunks["text"]])
q256 = np.array([U.embed(q["question"], dimensions=256) for q in qa_set])
dim_cmp = pd.DataFrame({"1024 dims": evaluate(chunks, base_matrix, q_vecs),
                        "256 dims": evaluate(chunks, m256, q256)}).T
dim_cmp["bytes_per_vector"] = [1024 * 4, 256 * 4]
dim_cmp.round(3)

Rule: the query and the documents must be embedded with the **same model and dimension**. If you
change either one you must re-embed and re-index everything.

### Query rewriting
Real users write vague or chatty questions. A small FM can rewrite the question into a clear search
query before it is embedded. Here are four messy questions and the document that answers each.

In [ ]:
messy = [("my kid is 22 and bumped the car, what do we pay ourselves", "03_claims_process.md"),
         ("windscreen chipped again!! will i lose my discount", "01_coverage_overview.md"),
         ("i do food delivery with my own car sometimes, is that a problem", "02_exclusions.md"),
         ("how do you people decide a claim looks fishy", "04_fraud_policy.md")]

REWRITE = ("Rewrite the customer message as one clear search query for a motor insurance policy manual. "
           "Use formal insurance terms. Return only the query.\n\nMessage: {q}")
rewritten = [U.converse(REWRITE.format(q=q), model_id=U.SMALL_MODEL, max_tokens=60, temperature=0) for q, _ in messy]
pd.DataFrame({"original": [q for q, _ in messy], "rewritten": rewritten})

Compare the rank of the first chunk from the correct document, before and after rewriting (1 is best).

In [ ]:
def first_rank(query, source):
    order, _ = rag.top_k(U.embed(query), base_matrix, k=len(chunks))
    return next(pos for pos, i in enumerate(order, start=1) if chunks["source"].iloc[i] == source)

pd.DataFrame({"question": [q for q, _ in messy],
              "rank_original": [first_rank(q, s) for q, s in messy],
              "rank_rewritten": [first_rank(r, s) for r, (_, s) in zip(rewritten, messy)]})

### Embedding type: float vs binary
Titan Text Embeddings V2 can also return **binary** embeddings (1 bit per dimension, 32x smaller).
Does retrieval suffer? Embed the heading chunks and the questions as bits.

In [ ]:
def embed_bits(text):
    body = json.dumps({"inputText": text, "dimensions": 1024, "embeddingTypes": ["binary"]})
    out = json.loads(U.client("bedrock-runtime").invoke_model(modelId=U.EMBED_MODEL, body=body)["body"].read())
    return np.array(out["embeddingsByType"]["binary"]) > 0

doc_bits = np.array([embed_bits(t) for t in chunks["text"]])
query_bits = np.array([embed_bits(q["question"]) for q in qa_set])
print(doc_bits.shape, "bits per chunk:", doc_bits.shape[1], "=", doc_bits.shape[1] // 8, "bytes")

Binary vectors are compared with Hamming distance. Trick: turn the bits into +1 / -1. Then the dot
product is `dimensions - 2 x Hamming distance`, so the same `evaluate` function gives the Hamming ranking.

In [ ]:
to_signs = lambda b: (2.0 * b - 1.0) / np.sqrt(b.shape[1])  # noqa: E731
type_cmp = pd.DataFrame({"float (1024 x 32 bit)": evaluate(chunks, base_matrix, q_vecs),
                         "binary (1024 x 1 bit)": evaluate(chunks, to_signs(doc_bits), to_signs(query_bits))}).T
type_cmp["bytes_per_vector"] = [1024 * 4, 1024 // 8]
type_cmp.round(3)

Typically there is a small drop for a 32x saving. A common design: binary search for a shortlist, then re-score
the shortlist with float vectors.

### Sparse vs dense vs hybrid retrieval
- **Dense** = the embeddings used so far (meaning).
- **Sparse** = one weight per vocabulary word (exact words). TF-IDF here; BM25 in a search engine.
- **Hybrid** = both. Simple version: add the two similarity scores.

Build TF-IDF vectors for the chunks and the questions.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words="english")
sparse_matrix = tfidf.fit_transform(chunks["text"]).toarray()
sparse_queries = tfidf.transform([q["question"] for q in qa_set]).toarray()
print("sparse shape:", sparse_matrix.shape, "| share of zeros:", round(float((sparse_matrix == 0).mean()), 3))

Hybrid by concatenation: if document and query vectors are joined `[dense, sparse]`, their dot
product is `dense score + sparse score`.

In [ ]:
hybrid_matrix = np.hstack([base_matrix, sparse_matrix])
hybrid_queries = np.hstack([q_vecs, sparse_queries])
mode_cmp = pd.DataFrame({"dense": evaluate(chunks, base_matrix, q_vecs),
                         "sparse (TF-IDF)": evaluate(chunks, sparse_matrix, sparse_queries),
                         "hybrid (dense + sparse)": evaluate(chunks, hybrid_matrix, hybrid_queries)}).T
mode_cmp.round(3)

The evaluation questions reuse many words from the documents, so sparse does well here. Now try the
four **messy** customer questions from above, which use different words.

In [ ]:
def first_rank_in(matrix, query_vector, source):
    order, _ = rag.top_k(query_vector, matrix, k=len(chunks))
    return next(pos for pos, i in enumerate(order, start=1) if chunks["source"].iloc[i] == source)

messy_dense = np.array([U.embed(q) for q, _ in messy])
messy_sparse = tfidf.transform([q for q, _ in messy]).toarray()
pd.DataFrame({"question": [q[:45] for q, _ in messy],
              "rank_dense": [first_rank_in(base_matrix, v, s) for v, (_, s) in zip(messy_dense, messy)],
              "rank_sparse": [first_rank_in(sparse_matrix, v, s) for v, (_, s) in zip(messy_sparse, messy)]})

Sparse search usually drops when the customer's words differ from the document's words, while dense
search holds up better. (If a question shares no words at all, its sparse rank is arbitrary.) That is why production RAG systems often run **hybrid** search (OpenSearch supports it natively; see
notebook 1.1b) and then a reranker.

### Hierarchical retrieval: search small chunks, return the parent
- Small chunks match a question precisely but give the model too little context.
- Large chunks give context but one vector has to stand for many facts.
- **Hierarchical (parent-child) retrieval** does both: embed small **children**, and return the
  **parent** section of the best child. Bedrock Knowledge Bases offer this as `HIERARCHICAL` chunking.

Parents = the heading chunks. Children = 200-character pieces of each parent.

In [ ]:
child_rows = [{"parent": p, "text": piece["text"]}
              for p, text in enumerate(chunks["text"])
              for piece in rag.chunk_fixed(text, "", size=200, overlap=40) if piece["text"].strip()]
children = pd.DataFrame(child_rows)
print(len(chunks), "parents,", len(children), "children")
children.head(3)

Embed the children (one call each).

In [ ]:
child_matrix = np.array([U.embed(t) for t in children["text"]])
child_matrix.shape

Rank **parents** by their best child: each parent's score is the highest score of any of its children.

In [ ]:
def parent_order(query_vector):
    child_scores = child_matrix @ query_vector
    parent_scores = np.full(len(chunks), -1.0)
    np.maximum.at(parent_scores, children["parent"].to_numpy(), child_scores)
    return np.argsort(-parent_scores)

Score it like before: where does the gold parent land for each question?

In [ ]:
def evaluate_orders(orders):
    ranks = np.array([next(pos for pos, i in enumerate(order, start=1) if i in gold_chunks(chunks, qa))
                      for qa, order in zip(qa_set, orders)])
    return {"recall@1": (ranks <= 1).mean(), "recall@3": (ranks <= 3).mean(), "mrr": (1 / ranks).mean()}

flat_orders = [rag.top_k(qv, base_matrix, k=len(chunks))[0] for qv in q_vecs]
hier_cmp = pd.DataFrame({"embed whole section, return section": evaluate_orders(flat_orders),
                         "embed children, return parent section": evaluate_orders([parent_order(qv) for qv in q_vecs])}).T
hier_cmp.round(3)

See it for one question: the child that matched, and the parent that would be sent to the model.

In [ ]:
qv = q_vecs[4]
best_child = int(np.argmax(child_matrix @ qv))
print("question   :", qa_set[4]["question"])
print("best child :", repr(children["text"][best_child][:110]))
print("its parent :", chunks["text"][children["parent"][best_child]][:260])

The SafeDrive sections are short, so here it may be no better, or even slightly worse, than embedding
the whole section. The pattern pays off with long documents, where a whole section is too big for one
good vector.

**More retrieval levers (managed options in Bedrock Knowledge Bases, hands-on in 3.2b)**

| Lever | Helps when |
|---|---|
| Hybrid search | Exact words, IDs, codes matter |
| Reranker model (`U.RERANK_MODEL`) | The right chunk is in the top 20 but not the top 3 |
| Metadata filter | Many similar documents (by year, product, Region) |
| Hierarchical / semantic chunking | Small chunks match well but lack context |
| Query decomposition | One question asks several things |
| Domain-specific or multilingual embedding model | General embeddings miss domain terms or languages |

## Exam traps

- The model never executes a tool. `stopReason = tool_use` means *your application* must run it and return a `toolResult`.
- `max_tokens` as the stop reason means the answer was cut off: raise `maxTokens`, do not lower the temperature.
- Fine-tuning does not add fresh facts cheaply and can cause forgetting; for changing knowledge use RAG.
- Changing the embedding model or its dimension requires re-embedding the whole corpus.
- Continued pre-training uses **unlabeled** data; fine-tuning uses **labeled** pairs. The exam swaps them.

## Quiz

**Q1.** An FM returns product categories in slightly different formats on every call, which breaks a parser.
Which is the quickest fix?
A) Fine-tune the model  B) Add few-shot examples and ask for a fixed JSON format, with temperature 0  C) Use a larger model  D) Increase `maxTokens`

<details><summary>Answer</summary>B. Examples plus an explicit format and low temperature fix most format problems without training.</details>

**Q2.** After fine-tuning an FM on legal contracts, it writes good contract clauses but has become poor at
general summaries. What happened and what helps?
A) Underfitting; train longer  B) Catastrophic forgetting; use LoRA adapters or mix general data into training
C) Data drift; retrain weekly  D) Hallucination; raise temperature

<details><summary>Answer</summary>B. The narrow training overwrote general abilities. Adapters, lower learning rate and replay data reduce this.</details>

**Q3.** A Converse API response has `stopReason: "tool_use"`. What must the application do next?
A) Retry with a higher temperature  B) Run the requested tool and send a `toolResult` message back
C) Nothing, Bedrock runs the tool  D) Increase `maxTokens`

<details><summary>Answer</summary>B. With the Converse API the caller executes the tool and returns the result in the next request.</details>

**Q4.** A RAG system retrieves the correct passage in the top 20 results for most questions, but rarely in
the top 3 that are sent to the model. Which change targets this problem?
A) Smaller embedding dimension  B) Add a reranker on the top 20  C) Lower the temperature  D) Continued pre-training

<details><summary>Answer</summary>B. A reranker reorders the candidates so the best ones reach the top positions.</details>

**Q5.** A company has 2 GB of unlabeled internal engineering reports and wants the FM to understand its
internal vocabulary. Which customization fits?
A) Fine-tuning  B) Continued pre-training  C) Distillation  D) Few-shot prompting

<details><summary>Answer</summary>B. Continued pre-training adapts a model with unlabeled domain text.</details>

## Save state
Record the retrieval results and where the Nova-format fine-tuning file is.

In [ ]:
state = U.save_state(finetune_nova_s3=U.s3_uri(FT_KEY),
                     retrieval_baseline={k: round(float(v), 3) for k, v in evaluate(chunks, base_matrix, q_vecs).items()})
state["retrieval_baseline"]

## Cleanup
Bedrock on-demand calls leave nothing running. The small fine-tuning file in S3 is kept (free tier
size). Only if you ran the 🔒 fine-tuning job is there something to delete: the custom model (it has a
monthly storage cost). Stop the job first if it is still running.

### Before you delete: what later notebooks use

**Nothing from this notebook is needed later.** It is safe to delete everything it created.

In [ ]:
if U.expensive("stop fine-tuning job / delete custom model"):
    U.safe(bedrock.stop_model_customization_job, jobIdentifier=FT_JOB)
    U.safe(bedrock.delete_custom_model, modelIdentifier="safedrive-claims-nova-micro")